# TCC — experimento com todos os 331 pares

Anexe o dataset original TCC RGI Guaxupé v1 e historico_125_pares.zip como entradas privadas. Ative Internet e GPU T4. O treino usa 296 pares. A validação usa 18 históricos e o teste reservado contém 17 históricos. A ordem B02/B03/B04/B08 é assumida com base no pipeline de exportação. Esta divisão é nova e usa os antigos recortes de avaliação no treino.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

CODE_REF = "fce8437a03bacf9ee5f1712d8812951b0a53ff81"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "tcc_base_ampliada"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/oguel/tcc-umamba.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Preserve as alterações no checkout antes de continuar.")
subprocess.run(["git", "fetch", "origin", CODE_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_REF
print("Código fixado:", CODE_REF)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-runtime.txt"), "geopandas"], check=True)
subprocess.run([sys.executable, "-c", "from src.models.umamba_runtime import install_prebuilt_colab_stack; print('PyTorch alterado:', install_prebuilt_colab_stack())"], cwd=REPO, check=True)


In [ ]:
DATA_ROOT = INPUT
image_folders = list(DATA_ROOT.rglob("Imagens_Individuais"))
if not image_folders:
    import hashlib
    import zipfile
    archives = list(DATA_ROOT.rglob("dataset_rgi_v1_privado.zip"))
    if len(archives) != 1:
        raise RuntimeError("Anexe o dataset privado com os GeoTIFFs ou o ZIP preparado.")
    digest = hashlib.sha256()
    with archives[0].open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == "9cae74a47fcc68885f2703a5bf3cd5d9230eadc400a9c5a57c3ed751376d60b4", "ZIP diferente"
    destination = WORK / "dataset_rgi_v1"
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Caminho inválido no ZIP")
        archive.extractall(destination)
    DATA_ROOT = destination
print("Raiz de dados:", DATA_ROOT)


In [ ]:
import os

EPOCHS = 100
original_hash = "1bc14d3a181adba0c3d0d1bae09e49376babd7dcab28a5e442820d9806e43952"
package_hash = original_hash
matches = sorted(INPUT.rglob("historico_125_pares.zip"))
if len(matches) > 1:
    raise RuntimeError("Anexe apenas uma copia do ZIP historico.")
temporary = None
if matches:
    package = matches[0]
else:
    from tempfile import TemporaryDirectory

    folders = sorted(p.parent for p in INPUT.rglob("divisao.csv")
                     if (p.parent / "images").is_dir() and (p.parent / "masks").is_dir())
    if len(folders) != 1:
        raise RuntimeError("Anexe o ZIP historico ou uma unica pasta extraida com images, masks e divisao.csv.")
    folder = folders[0]
    files = sorted([folder / "divisao.csv", *folder.glob("images/*.npy"), *folder.glob("masks/*.npy")])
    if len(files) != 251:
        raise RuntimeError("Esperados 125 imagens, 125 mascaras e divisao.csv.")
    temporary = TemporaryDirectory(dir=WORK)
    package = Path(temporary.name) / "historico_125_pares.zip"
    signature = hashlib.sha256()
    with zipfile.ZipFile(package, "w") as archive:
        for file in files:
            name = file.relative_to(folder).as_posix()
            data = file.read_bytes()
            signature.update(name.encode() + b"\x00" + hashlib.sha256(data).digest())
            archive.writestr(name, data)
    if signature.hexdigest() != "4356288f394c2954c581537107b5e0a7ce141707d42fac866a01addd4f4e81df":
        raise RuntimeError("Os arquivos extraidos diferem do pacote auditado.")
    with package.open("rb") as stream:
        package_hash = hashlib.file_digest(stream, "sha256").hexdigest()
script = WORK / "train_todos_pares.py"
script.write_text('"""Treino controlado com o suplemento historico e a divisao espacial original."""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset\n\nfrom scripts.train_emater_rgi import build_model\nfrom src.data.augmentation import RGIPatchViews\nfrom src.data.emater_dataset import EMATERRGIDataset, find_rgi_pairs\nfrom src.data.rgi_fingerprint import dataset_fingerprint\nfrom src.losses import BCEDiceLoss\nfrom src.trainer import fit_model\nfrom src.training_state import save_json\n\nSUPPLEMENT_SHA256 = "1bc14d3a181adba0c3d0d1bae09e49376babd7dcab28a5e442820d9806e43952"\nOFFICIAL_SHA256 = "5a0f95d581f65ed1533caea509cdc04bc7d348b9ce36cf6d91181196328f58b0"\n\n\nclass HistoricalPairs(Dataset):\n    def __init__(self, archive):\n        self.archive = archive\n        images = {name.removeprefix("images/") for name in archive.namelist() if name.startswith("images/")}\n        masks = {name.removeprefix("masks/") for name in archive.namelist() if name.startswith("masks/")}\n        if images != masks or len(images) != 125:\n            raise ValueError("Suplemento sem os 125 pares esperados.")\n        self.names = sorted(images)\n\n    def __len__(self):\n        return len(self.names)\n\n    def __getitem__(self, index):\n        name = self.names[index]\n        image = np.load(io.BytesIO(self.archive.read("images/" + name)), allow_pickle=False)\n        mask = np.load(io.BytesIO(self.archive.read("masks/" + name)), allow_pickle=False)\n        if (\n            image.shape != (4, 512, 512) or image.dtype != np.float32\n            or not np.isfinite(image).all() or image.min() < 0 or image.max() > 1\n            or mask.shape != (512, 512) or mask.dtype != np.uint8\n            or not np.isin(mask, [0, 1, 255]).all() or np.all(mask == 255)\n        ):\n            raise ValueError(f"Par historico invalido: {name}")\n        return torch.from_numpy(image), torch.from_numpy(mask.astype(np.float32)[None])\n\n\ndef build_sources(root, archive):\n    pairs = find_rgi_pairs(root)\n    if dataset_fingerprint(pairs) != OFFICIAL_SHA256:\n        raise ValueError("A base atual difere da versao auditada.")\n    current = EMATERRGIDataset(pairs)\n    historical = HistoricalPairs(archive)\n    manifest = list(csv.DictReader(io.StringIO(archive.read("divisao.csv").decode("utf-8"))))\n    current_ids = {f"{p.municipality}/{p.image.name}" for p in pairs}\n    assert {r["patch"] for r in manifest if r["origem"] == "base_atual" and r["split"] == "train"} == current_ids\n    historical_roles = {r["patch"]: r["split"] for r in manifest if r["origem"] == "historico"}\n    assert set(historical_roles) == set(historical.names) and len(manifest) == 331\n    partitions = {role: [i for i, name in enumerate(historical.names) if historical_roles[name] == role]\n                  for role in ("train", "validation", "test_locked")}\n    assert tuple(map(len, partitions.values())) == (90, 18, 17)\n    return (ConcatDataset([current, Subset(historical, partitions["train"])]),\n            Subset(historical, partitions["validation"]))\n\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", required=True, type=Path)\n    parser.add_argument("--supplement", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--epochs", type=int, default=100)\n    args = parser.parse_args()\n    if args.epochs < 1:\n        parser.error("Informe ao menos uma epoca.")\n    if not torch.cuda.is_available() or torch.__version__ != "2.10.0+cu126":\n        raise RuntimeError("Execute com GPU e PyTorch 2.10.0+cu126.")\n    with args.supplement.open("rb") as stream:\n        if hashlib.file_digest(stream, "sha256").hexdigest() != SUPPLEMENT_SHA256:\n            raise ValueError("O ZIP historico difere do suplemento auditado.")\n    from src.models.umamba_runtime import ensure_umamba_runtime\n\n    runtime = ensure_umamba_runtime()\n    device = torch.device("cuda")\n    with ZipFile(args.supplement) as archive:\n        train, validation = build_sources(args.root, archive)\n        args.output.mkdir(parents=True, exist_ok=False)\n        for name in ("unet", "umamba"):\n            random.seed(42)\n            np.random.seed(42)\n            torch.manual_seed(42)\n            torch.backends.cudnn.benchmark = False\n            train_views = RGIPatchViews(train, size=256, mode="train", augment=True, train_views_per_image=4)\n            val_views = RGIPatchViews(validation, size=256, mode="eval")\n            train_loader = DataLoader(train_views, batch_size=1, shuffle=True, pin_memory=True,\n                                      generator=torch.Generator().manual_seed(42))\n            val_loader = DataLoader(val_views, batch_size=1, pin_memory=True,\n                                    generator=torch.Generator().manual_seed(43))\n            model = build_model(name, 256).to(device)\n            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)\n            folder = args.output / name\n            contract = {\n                "model": name, "epochs": args.epochs, "seed": 42, "patch_size": 256,\n                "batch_size": 1, "train_patches": len(train), "validation_patches": len(validation),\n                "train_views": len(train_views), "validation_views": len(val_views),\n                "base_sha256": OFFICIAL_SHA256, "historical_zip_sha256": SUPPLEMENT_SHA256,\n                "historical_band_order": ["B02", "B03", "B04", "B08"],\n                "test_evaluated": False, "runtime": runtime,\n                "runner_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n            }\n            save_json(folder / "invocation.json", contract)\n            fit_model(model, train_loader, val_loader, BCEDiceLoss(ignore_index=255),\n                      optimizer, device, args.epochs, folder / f"best_{name}_ampliada.pt",\n                      run_contract=contract)\n            save_json(folder / "conclusao.json", {"epochs_completed": args.epochs, "test_evaluated": False})\n            del model, optimizer\n\n\nif __name__ == "__main__":\n    main()\n'.replace(original_hash, package_hash), encoding="utf-8")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = WORK / f"treino_todos_pares_{stamp}"
subprocess.run([sys.executable, "-u", str(script), "--root", str(DATA_ROOT),
    "--supplement", str(package), "--output", str(output), "--epochs", str(EPOCHS)],
    cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO)}, check=True)
if temporary is not None:
    temporary.cleanup()
print("BAIXAR:", shutil.make_archive(str(output), "zip", root_dir=output))
